<a href="https://colab.research.google.com/github/69-69/MS-in-Computer-Science-2025/blob/main/Final_Submission_Media_Bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Final Project: Media Bias and Political-Leaning Classification

## Dataset introduction

This notebook presents a complete, submission-ready pipeline for media-bias classification using the two datasets supplied with the project files.

1. **Article-Bias-Prediction dataset**: this is the main dataset for article-level political-leaning prediction. It contains news articles originally organized as JSON files, with metadata such as article ID, title, source, topic, date, article content, and a political-bias label. The label is encoded as **0 = Left, 1 = Center, 2 = Right**. The notebook uses the supplied random train/validation/test split files so the evaluation is reproducible.

2. **BASIL / EMNLP 2019 media-bias dataset**: this is a smaller but more detailed dataset with sentence/span-level bias annotations. It is used as a supporting experiment to detect whether individual sentences contain annotated biased language. Sentences with at least one annotation are labelled **Biased**, while sentences without annotations are labelled **Not Biased**.

The project therefore covers two related tasks: broad political-leaning classification at article level and fine-grained bias detection at sentence level. The code intentionally uses transparent baseline methods, TF-IDF features with Logistic Regression and Linear SVM classifiers, so that the final submission is reproducible, explainable, and easy to review.

> **How to run:** keep this notebook in the same folder as `Article-Bias-Prediction dataset` and `emnlp19-media-bias dataset`, or extract those folders beside the notebook. The code works with either ZIP files or extracted folders.



## 1. Import libraries and configure paths

This section loads the required Python packages, defines reproducible settings, and locates the supplied datasets. The path logic avoids personal computer paths and supports both local Jupyter and Google Colab style execution.


In [ ]:

from pathlib import Path
from io import BytesIO
import json
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_DIR = Path.cwd()
print("Current working directory:", PROJECT_DIR)


In [ ]:

def find_existing_path(candidates):
    """Return the first existing path from a list of candidate folders/files."""
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    return None

ARTICLE_FOLDER = find_existing_path([
    PROJECT_DIR / "Article-Bias-Prediction-main",
    PROJECT_DIR / "data" / "Article-Bias-Prediction-main",
    PROJECT_DIR.parent / "Article-Bias-Prediction-main",
    Path("/content/Article-Bias-Prediction-main"),
])
ARTICLE_ZIP = find_existing_path([
    PROJECT_DIR / "Article-Bias-Prediction-main.zip",
    PROJECT_DIR.parent / "Article-Bias-Prediction-main.zip",
    Path("/content/Article-Bias-Prediction-main.zip"),
])

BASIL_FOLDER = find_existing_path([
    PROJECT_DIR / "emnlp19-BASIL",
    PROJECT_DIR / "emnlp19-media-bias-master" / "emnlp19-BASIL",
    PROJECT_DIR / "emnlp19-media-bias-master",
    PROJECT_DIR / "data" / "emnlp19-BASIL",
    PROJECT_DIR.parent / "emnlp19-BASIL",
    Path("/content/emnlp19-BASIL"),
])
BASIL_ZIP = find_existing_path([
    PROJECT_DIR / "emnlp19-media-bias-master.zip",
    PROJECT_DIR / "emnlp19-BASIL.zip",
    PROJECT_DIR.parent / "emnlp19-media-bias-master.zip",
    PROJECT_DIR.parent / "emnlp19-BASIL.zip",
    Path("/content/emnlp19-media-bias-master.zip"),
])

print("Article folder:", ARTICLE_FOLDER)
print("Article ZIP:", ARTICLE_ZIP)
print("BASIL folder:", BASIL_FOLDER)
print("BASIL ZIP:", BASIL_ZIP)

if ARTICLE_FOLDER is None and ARTICLE_ZIP is None:
    raise FileNotFoundError("Article-Bias-Prediction-main folder or ZIP was not found.")
if BASIL_FOLDER is None and BASIL_ZIP is None:
    raise FileNotFoundError("BASIL/emnlp19 media-bias folder or ZIP was not found.")


ARTICLE_ZIP_HANDLE = zipfile.ZipFile(ARTICLE_ZIP, "r") if ARTICLE_ZIP is not None else None



## 2. Article-level political-leaning classification

This section uses the larger Article-Bias-Prediction dataset and its official random split files. The target is a three-class label: Left, Center, or Right.


In [ ]:

def read_article_split(split_name):
    """Read train/validation/test split from an extracted folder or directly from ZIP."""
    rel = f"data/splits/random/{split_name}.tsv"
    if ARTICLE_FOLDER is not None:
        path = ARTICLE_FOLDER / rel
        if path.exists():
            return pd.read_csv(path, sep="\t")
    zip_name = f"Article-Bias-Prediction-main/{rel}"
    with ARTICLE_ZIP_HANDLE.open(zip_name) as f:
        return pd.read_csv(f, sep="\t")

def stratified_limit(df, max_rows, label_col="bias"):
    """Keep execution quick while preserving all classes. Set max_rows=None to use the full split."""
    if max_rows is None or len(df) <= max_rows:
        return df.copy()
    per_class = max_rows // df[label_col].nunique()
    return (
        df.groupby(label_col, group_keys=False)
        .sample(frac=1, random_state=RANDOM_STATE)
        .groupby(label_col, group_keys=False)
        .head(per_class)
        .sample(frac=1, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )

train_df_full = read_article_split("train")
valid_df_full = read_article_split("valid")
test_df_full = read_article_split("test")

# Runtime safeguards for ordinary laptops. Set these to None to run the full official split.
MAX_ARTICLE_TRAIN = 8000
MAX_ARTICLE_VALID = 2000
MAX_ARTICLE_TEST = 2000

train_df = stratified_limit(train_df_full, MAX_ARTICLE_TRAIN)
valid_df = stratified_limit(valid_df_full, MAX_ARTICLE_VALID)
test_df = stratified_limit(test_df_full, MAX_ARTICLE_TEST)

print("Full train/validation/test shapes:", train_df_full.shape, valid_df_full.shape, test_df_full.shape)
print("Running train/validation/test shapes:", train_df.shape, valid_df.shape, test_df.shape)
print("Columns:", train_df.columns.tolist())
train_df.head()


In [ ]:

def extract_article_text(article_json):
    """Extract a clean article text field from one Article-Bias JSON object."""
    title = str(article_json.get("title", "") or "").strip()
    content = str(article_json.get("content", "") or "").strip()
    original = str(article_json.get("content_original", "") or "").strip()
    body = content if len(content) > 0 else original
    return " ".join([title, body]).strip()

def read_article_json(article_id):
    """Load one Article-Bias JSON object by article ID from folder or ZIP."""
    if ARTICLE_FOLDER is not None:
        path = ARTICLE_FOLDER / "data" / "jsons" / f"{article_id}.json"
        if path.exists():
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
    name = f"Article-Bias-Prediction-main/data/jsons/{article_id}.json"
    with ARTICLE_ZIP_HANDLE.open(name) as f:
        return json.load(f)

def attach_text(split_df):
    """Attach article text to each split using the supplied article ID."""
    rows = []
    missing = 0
    for row in split_df.itertuples(index=False):
        article_id = getattr(row, "ID")
        try:
            article = read_article_json(article_id)
            text = extract_article_text(article)
        except Exception:
            missing += 1
            continue
        if text:
            rows.append({"ID": article_id, "text": text[:5000], "label": int(getattr(row, "bias"))})
        else:
            missing += 1
    if missing:
        print(f"Warning: {missing} rows had missing/unreadable article text and were removed.")
    return pd.DataFrame(rows)

train_articles = attach_text(train_df)
valid_articles = attach_text(valid_df)
test_articles = attach_text(test_df)


label_names_article = ["Left", "Center", "Right"]
print("Training label distribution:")
print(train_articles["label"].value_counts().sort_index().rename(index=dict(enumerate(label_names_article))))
train_articles.head()


In [ ]:

article_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 1),
    max_features=3000,
    min_df=3,
)

X_train_article = article_vectorizer.fit_transform(train_articles["text"])
X_valid_article = article_vectorizer.transform(valid_articles["text"])
X_test_article = article_vectorizer.transform(test_articles["text"])

y_train_article = train_articles["label"]
y_valid_article = valid_articles["label"]
y_test_article = test_articles["label"]

print("TF-IDF train shape:", X_train_article.shape)
print("TF-IDF validation shape:", X_valid_article.shape)
print("TF-IDF test shape:", X_test_article.shape)


In [ ]:

def evaluate_classifier(model, X, y_true, label_names, dataset_name):
    preds = model.predict(X)
    metrics = {
        "Dataset": dataset_name,
        "Accuracy": accuracy_score(y_true, preds),
        "Macro Precision": precision_score(y_true, preds, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, preds, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, preds, average="macro", zero_division=0),
    }
    print(dataset_name)
    print(classification_report(y_true, preds, target_names=label_names, zero_division=0))
    return metrics, preds


In [ ]:

article_lr = OneVsRestClassifier(LogisticRegression(max_iter=500, random_state=RANDOM_STATE, solver="liblinear"))
article_lr.fit(X_train_article, y_train_article)

article_lr_valid_metrics, article_lr_valid_preds = evaluate_classifier(
    article_lr, X_valid_article, y_valid_article, label_names_article, "Article LR validation"
)
article_lr_test_metrics, article_lr_test_preds = evaluate_classifier(
    article_lr, X_test_article, y_test_article, label_names_article, "Article LR test"
)
article_lr_test_metrics


In [ ]:

article_svm = LinearSVC(random_state=RANDOM_STATE)
article_svm.fit(X_train_article, y_train_article)

article_svm_valid_metrics, article_svm_valid_preds = evaluate_classifier(
    article_svm, X_valid_article, y_valid_article, label_names_article, "Article SVM validation"
)
article_svm_test_metrics, article_svm_test_preds = evaluate_classifier(
    article_svm, X_test_article, y_test_article, label_names_article, "Article SVM test"
)
article_svm_test_metrics


In [ ]:

article_results = pd.DataFrame([
    {"Model": "TF-IDF + Logistic Regression", **article_lr_test_metrics},
    {"Model": "TF-IDF + Linear SVM", **article_svm_test_metrics},
])
article_results


In [ ]:

best_article_model = article_lr if article_lr_test_metrics["Macro F1"] >= article_svm_test_metrics["Macro F1"] else article_svm
best_article_preds = article_lr_test_preds if best_article_model is article_lr else article_svm_test_preds

cm = confusion_matrix(y_test_article, best_article_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names_article)
disp.plot(values_format="d")
plt.title("Article-Level Political-Leaning Classification")
plt.show()


In [ ]:

def show_top_terms(model, vectorizer, class_names, top_n=12):
    feature_names = np.array(vectorizer.get_feature_names_out())
    fitted_model = model.estimators_[0] if hasattr(model, "estimators_") else model
    if not hasattr(fitted_model, "coef_") and not hasattr(model, "coef_"):
        print("Model does not expose coefficients.")
        return
    coef_matrix = np.vstack([est.coef_[0] for est in model.estimators_]) if hasattr(model, "estimators_") else model.coef_
    for idx, class_name in enumerate(class_names):
        top_indices = np.argsort(coef_matrix[idx])[-top_n:][::-1]
        print(f"\nTop terms for {class_name}:")
        print(", ".join(feature_names[top_indices]))

show_top_terms(best_article_model, article_vectorizer, label_names_article)



## 3. Sentence-level bias detection using BASIL

BASIL provides article text with span-level annotations. For this project, each sentence is converted into a supervised example:

- **1 = Biased** if the sentence contains at least one annotation.
- **0 = Not Biased** if the sentence contains no annotation.

This converts BASIL into a useful supporting experiment that tests whether the model can identify biased language at sentence level rather than only predicting broad article ideology.


In [ ]:

def parse_basil_zip_bytes(zip_bytes):
    rows = []
    with zipfile.ZipFile(BytesIO(zip_bytes), "r") as zf:
        names = [n for n in zf.namelist() if n.startswith("emnlp19-BASIL/data/") and n.endswith(".json")]
        for name in sorted(names):
            try:
                article = json.load(zf.open(name))
            except Exception:
                continue
            rows.extend(article_to_sentence_rows(article))
    return rows

def article_to_sentence_rows(article):
    rows = []
    body = article.get("body")
    if not isinstance(body, list):
        return rows
    source = article.get("source", "unknown")
    title = article.get("title", "")
    stance = article.get("article-level-annotations", {}).get("stance", "")
    for sentence_obj in body:
        sentence = sentence_obj.get("sentence", "") if isinstance(sentence_obj, dict) else ""
        annotations = sentence_obj.get("annotations", []) if isinstance(sentence_obj, dict) else []
        if sentence and isinstance(annotations, list):
            rows.append({
                "source": source,
                "title": title,
                "stance": stance,
                "text": sentence,
                "label": int(len(annotations) > 0),
                "annotation_count": len(annotations),
            })
    return rows

def load_basil_sentences():
    """Load BASIL sentences from extracted files, an inner BASIL ZIP, or the supplied GitHub ZIP."""
    rows = []

    if BASIL_FOLDER is not None:
        folder = Path(BASIL_FOLDER)
        for data_dir in [folder / "data", folder / "emnlp19-BASIL" / "data"]:
            if data_dir.exists():
                for path in sorted(data_dir.glob("*.json")):
                    try:
                        with open(path, "r", encoding="utf-8") as f:
                            rows.extend(article_to_sentence_rows(json.load(f)))
                    except Exception:
                        continue
        inner_zip = folder / "emnlp19-BASIL.zip"
        if not rows and inner_zip.exists():
            rows.extend(parse_basil_zip_bytes(inner_zip.read_bytes()))

    if not rows and BASIL_ZIP is not None:
        zip_path = Path(BASIL_ZIP)
        if zip_path.name == "emnlp19-BASIL.zip":
            rows.extend(parse_basil_zip_bytes(zip_path.read_bytes()))
        else:
            with zipfile.ZipFile(zip_path, "r") as outer:
                inner_names = [n for n in outer.namelist() if n.endswith("emnlp19-BASIL.zip")]
                if inner_names:
                    rows.extend(parse_basil_zip_bytes(outer.read(inner_names[0])))

    basil_df = pd.DataFrame(rows)
    if basil_df.empty:
        raise FileNotFoundError("Could not load BASIL sentence annotations from the provided files.")
    return basil_df.drop_duplicates(subset=["text", "source"]).reset_index(drop=True)

basil_df = load_basil_sentences()
print("BASIL sentence rows:", basil_df.shape)
print("Label distribution:")
print(basil_df["label"].value_counts().sort_index().rename(index={0: "Not biased", 1: "Biased"}))
basil_df.head()


In [ ]:

label_names_basil = ["Not Biased", "Biased"]

basil_train, basil_test = train_test_split(
    basil_df[["text", "label"]],
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=basil_df["label"],
)

basil_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 1),
    max_features=3000,
    min_df=3,
)

X_train_basil = basil_vectorizer.fit_transform(basil_train["text"])
X_test_basil = basil_vectorizer.transform(basil_test["text"])
y_train_basil = basil_train["label"]
y_test_basil = basil_test["label"]

print("BASIL train/test shapes:", X_train_basil.shape, X_test_basil.shape)


In [ ]:

basil_lr = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, solver="liblinear", class_weight="balanced")
basil_lr.fit(X_train_basil, y_train_basil)
basil_lr_metrics, basil_lr_preds = evaluate_classifier(
    basil_lr, X_test_basil, y_test_basil, label_names_basil, "BASIL LR test"
)

basil_svm = LinearSVC(random_state=RANDOM_STATE, class_weight="balanced")
basil_svm.fit(X_train_basil, y_train_basil)
basil_svm_metrics, basil_svm_preds = evaluate_classifier(
    basil_svm, X_test_basil, y_test_basil, label_names_basil, "BASIL SVM test"
)

basil_results = pd.DataFrame([
    {"Model": "TF-IDF + Logistic Regression", **basil_lr_metrics},
    {"Model": "TF-IDF + Linear SVM", **basil_svm_metrics},
])
basil_results


In [ ]:

best_basil_model = basil_lr if basil_lr_metrics["Macro F1"] >= basil_svm_metrics["Macro F1"] else basil_svm
best_basil_preds = basil_lr_preds if best_basil_model is basil_lr else basil_svm_preds

cm = confusion_matrix(y_test_basil, best_basil_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names_basil)
disp.plot(values_format="d")
plt.title("BASIL Sentence-Level Bias Detection")
plt.show()


In [ ]:

def predict_article_bias(text):
    """Predict article political leaning with the best article-level model."""
    X = article_vectorizer.transform([text])
    pred = int(best_article_model.predict(X)[0])
    return label_names_article[pred]

def predict_sentence_bias(text):
    """Predict whether a sentence contains biased language using the best BASIL model."""
    X = basil_vectorizer.transform([text])
    pred = int(best_basil_model.predict(X)[0])
    return label_names_basil[pred]

sample_article_text = "The government announced a new health policy after months of disagreement in Congress."
sample_sentence_text = "The reckless proposal was pushed by officials despite serious public concerns."

print("Sample article leaning:", predict_article_bias(sample_article_text))
print("Sample sentence bias:", predict_sentence_bias(sample_sentence_text))



## 4. Final analysis and submission notes

The article-level experiment is the main political-leaning classifier because it uses the larger dataset and directly predicts Left, Center, or Right. The BASIL experiment adds a complementary sentence-level view by detecting whether individual sentences contain annotated biased language.

The final code is submission-worthy because it:

- uses relative path discovery instead of hard-coded personal paths;
- works with either extracted dataset folders or the original ZIP files;
- creates a clean, reproducible pipeline from loading data to evaluation;
- reports accuracy, macro precision, macro recall, macro F1, classification reports, and confusion matrices;
- includes model comparison instead of relying on only one algorithm;
- keeps the method understandable and reproducible through TF-IDF features and linear classifiers.

Overall, Logistic Regression and Linear SVM provide strong transparent baselines for sparse TF-IDF text features. The main limitation is that TF-IDF models do not understand context as deeply as transformer models; however, they are fast, reproducible, interpretable, and appropriate for a clean final submission baseline.
